# Phase 6 — Suivi des portefeuilles

## Pourquoi cette phase est importante

Les trois portefeuilles sont déjà construits. Ils partagent les mêmes titres et les mêmes rendements. Cette phase regarde où va le poids, et à quel point il est concentré. Elle ne recalcule pas les poids.

Le suivi répond à trois questions.

- Quelle part du portefeuille est dans chaque secteur, chaque titre et chaque zone.
- Quelle part est en actions, en obligations et en ETF.
- Quelle part tiennent les dix premières lignes, et le risque de concentration qui en résulte.

## Les livrables

| Livrable | Fichier | Rôle |
|---|---|---|
| Documentation | `documentation/06_portfolio_monitoring.ipynb` | La méthode, les formules, les fichiers produits |
| Rapport mensuel | `reports/monitoring/06_rapport_mensuel.ipynb` | La photo des trois portefeuilles au dernier mois de la fenêtre |
| Dashboard | `notebooks/06_dashboard_monitoring.ipynb` et `reports/monitoring/dashboard.html` | Les graphiques des trois portefeuilles |

La vérification automatique est à part, dans `tests/monitoring/phase_06_validation.ipynb`. Elle ne remplace aucun livrable.

## Une photo, pas un historique de mandat

Les poids du minimum de volatilité et du Sharpe maximal sont estimés sur toute la fenêtre. Le rapport porte la date du dernier mois de cette fenêtre, lue dans `portfolio_metrics.csv`. Quand un nouveau mois de SARON arrive, la phase 5 réestime les poids et cette phase remplace la photo.

Le fichier des rendements mensuels applique ces derniers poids à tous les mois de la fenêtre. Le suivi ne le lit pas comme l'allocation qu'un mandat aurait eue au premier mois.


## Entrées

Le module lit les poids publiés et la classification du fichier de positions du benchmark. Il ne réécrit ni les cours ni les poids.

| Rôle | Fichier | Colonnes utilisées |
|---|---|---|
| Poids | `data/04_portfolios/portfolio_weights.csv` | `strategy`, `yahoo_symbol`, `ticker`, `name`, `weight` |
| Fenêtre | `data/04_portfolios/portfolio_metrics.csv` | `strategy`, `names`, `window_start`, `window_end`, `maximum_weight`, `effective_holdings` |
| Classification | feuille de positions du benchmark, retrouvée par le jeu qui produit `security_candidates.csv` | `Ticker`, `Sector`, `Asset Class`, `Location` |

Aucun identifiant de jeu n'est écrit dans le module. Le fichier de positions est celui que le registre désigne déjà comme entrée des candidats. Si `Ticker` n'est pas unique dans cette feuille, le calcul s'arrête : un même ticker ne peut pas porter deux secteurs.

Un titre du portefeuille absent de cette feuille reste dans le portefeuille. Son secteur, son pays et sa classe d'actif deviennent `Non classé`. Le poids n'est pas retiré, et les totaux continuent de faire 1.

## Secteur, zone, exposition

Le secteur est le texte de la colonne `Sector`, sans traduction. Une valeur vide devient `Non classé`.

Le pays est le texte de `Location`. Une valeur vide, ou `--`, devient `Non classé`. La zone est un regroupement de ce pays :

| Zone | Pays publiés par la feuille de positions |
|---|---|
| Asie-Pacifique | Australia, China, Hong Kong, India, Japan, Korea (South), Malaysia, Singapore, Taiwan, Thailand |
| Europe | Austria, Belgium, Denmark, European Union, France, Germany, Italy, Norway, Portugal, Spain, Sweden, Switzerland, United Kingdom |
| Amériques | Brazil, Canada, Chile, Mexico, United States |
| Moyen-Orient et Afrique | Israel, Saudi Arabia, South Africa |

Un pays qui n'est pas dans cette table reste dans le portefeuille, dans la zone `Non classé`. Le journal technique le signale. Le calcul ne s'arrête pas, et le pays n'est pas rangé dans une zone au jugé.

L'exposition reprend `Asset Class`.

| Classe publiée | Exposition |
|---|---|
| Equity | Actions |
| Fixed Income, Bond, Bonds | Obligations |
| ETF | ETF |

Toute autre classe garde son nom de source, par exemple Cash ou Futures. Les trois lignes Actions, Obligations et ETF sont toujours écrites. Une ligne à 0 dit que le portefeuille n'a pas cette exposition. Elle n'est pas inventée pour remplir le tableau.

Une ligne dont le poids est inférieur ou égal à `1e-8` est de la poussière numérique de l'optimiseur, pas une position. Elle reste dans le fichier ligne à ligne et dans l'indice de Herfindahl. Elle sort des tableaux de secteur, de zone et d'exposition. Le seuil est très au-dessus de cette poussière, qui est de l'ordre de `1e-17`, et très au-dessous d'une ligne réellement publiée.


## Formules

Soit $w_i$ le poids publié du titre $i$, et $N$ le nombre de titres de l'univers commun. Ce nombre comprend les lignes que l'optimiseur a mises à zéro. Les trois portefeuilles ont le même $N$.

L'allocation d'un groupe $G$ (un secteur, une zone, ou une exposition) est

$$
w(G) = \sum_{i \in G} w_i
$$

Les groupes d'une même coupe se partagent le portefeuille. Leur somme vaut 1, à la poussière numérique près. Le module refuse de renormaliser un fichier de poids qui ne somme pas déjà à 1.

Les dix premières lignes sont les plus grands poids. À poids égal, l'ordre suit `yahoo_symbol`. S'il y a moins de dix lignes au-dessus du seuil, le top 10 est la somme de ces lignes.

$$
w_{(10)} = \sum_{k=1}^{\min(10, N_+)} w_{(k)}
$$

$N_+$ est le nombre de lignes au-dessus du seuil. Pour l'équipondéré, tous les poids sont égaux, donc $w_{(10)} = 10 / N$. Les dix noms affichés sont alors les dix premiers symboles, pas dix titres plus gros que les autres.

L'indice de Herfindahl utilise tous les poids publiés, y compris les zéros :

$$
H = \sum_{i=1}^{N} w_i^2
$$

Le nombre effectif de titres est

$$
N_{eff} = \frac{1}{H}
$$

Il vaut $N$ quand tous les poids sont égaux. Il baisse quand le poids se concentre. La phase 5 publie déjà $N_{eff}$. Cette phase le recalcule et s'arrête si les deux valeurs divergent.

Le multiple de concentration compare le portefeuille à l'équipondéré de même univers :

$$
C = N \, H
$$

$C = 1$ quand chaque titre pèse $1/N$. $C$ augmente quand le poids se concentre. Un multiple de 4 veut dire que le portefeuille est aussi concentré qu'un équipondéré de $N/4$ titres.


## Le chemin dans le temps

La photo décrit les poids du dernier mois. Le chemin décrit ce que ces mêmes poids ont produit pendant la fenêtre.

Les poids cibles sont remis au début de chaque mois. La part d'un secteur, d'une zone ou d'une exposition ne dérive donc pas. Un graphique d'allocation mois par mois serait une ligne plate. Le mouvement est ailleurs : le marché change, et le résultat du portefeuille avec lui.

Le rendement du titre $i$ au mois $t$ est $r_{i,t}$. Avec le poids cible $w_i$, sa contribution est

$$
c_{i,t} = w_i \, r_{i,t}
$$

La contribution d'un secteur est la somme des contributions de ses titres. Au mois $t$, la somme des secteurs est le rendement du portefeuille. La série cumulée additionne ces contributions. Elle n'est pas la valeur composée du portefeuille.

La valeur d'un franc part de 1 :

$$
W_t = \prod_{u \le t} (1 + R_u)
$$

La perte depuis le plus haut est

$$
D_t = \frac{W_t}{\max_{u \le t} W_u} - 1
$$

$R_u$ est le rendement du portefeuille au mois $u$. Ces deux séries, et les contributions, appliquent les poids estimés sur toute la fenêtre à chacun de ses mois. Elles ne décrivent pas un mandat dont les poids auraient été choisis au premier mois et laissés tels quels.

## Fichiers produits

Tous sont sous `data/05_monitoring/`. Chaque exécution les remplace.

| Fichier | Contenu |
|---|---|
| `allocation_by_holding.csv` | Une ligne par stratégie et par titre, avec secteur, pays, zone et exposition |
| `allocation_by_sector.csv` | Poids et nombre de lignes par secteur |
| `allocation_by_zone.csv` | Poids et nombre de lignes par zone |
| `exposure.csv` | Actions, obligations, ETF, puis toute autre classe présente |
| `concentration.csv` | Poids maximum, top 10, Herfindahl, nombre effectif, multiple |
| `top10_positions.csv` | Les dix premières lignes |
| `monthly_report.csv` | Une ligne par stratégie, datée du dernier mois de la fenêtre |
| `portfolio_path.csv` | Chaque mois : rendement, valeur d'un franc, perte depuis le plus haut |
| `sector_contribution.csv` | Contribution du secteur au rendement du mois, puis cumul |

La commande qui reconstruit ces fichiers seuls est

```
python -m src.portfolio_monitoring.build_monitoring
```

`python -m src.pipeline.refresh` le fait après la construction des portefeuilles, sur les poids qui viennent d'être estimés. La commande écrit aussi `reports/monitoring/dashboard.html`.

La cellule suivante lit le rapport déjà publié et affiche le mois, la fenêtre et l'exposition. Elle ne fixe aucune de ces valeurs.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import MONITORING_DATA

report = pd.read_csv(MONITORING_DATA / "monthly_report.csv")
row = report.iloc[0]
print(f"Mois du rapport : {row.report_month}")
print(f"Fenêtre : {row.window_start} à {row.window_end}")
print(f"Stratégies : {report['strategy'].nunique()}")
for item in report.itertuples(index=False):
    print(
        f"{item.strategy} : {int(item.invested_names)} lignes investies, "
        f"top 10 {item.top_10_weight:.1%}, "
        f"actions {item.actions_weight:.1%}, "
        f"obligations {item.obligations_weight:.1%}, "
        f"ETF {item.etf_weight:.1%}."
    )


Mois du rapport : 2026-09
Fenêtre : 2016-10 à 2026-09
Stratégies : 3
equal_weight : 83 lignes investies, top 10 12.0%, actions 100.0%, obligations 0.0%, ETF 0.0%.
minimum_volatility : 27 lignes investies, top 10 72.1%, actions 100.0%, obligations 0.0%, ETF 0.0%.
maximum_sharpe : 19 lignes investies, top 10 87.0%, actions 100.0%, obligations 0.0%, ETF 0.0%.
